In [2]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

from pyLIMA import event, telescopes
from astropy import units as u
from pyLIMA.models import FSPLarge_model, USBL_model

import ipywidgets as widgets
from IPython.display import display, clear_output
from pyLIMA.simulations import simulator
from pyLIMA.toolbox import brightness_transformation
current_path = os.getcwd()
parent_directory = os.path.abspath(os.path.join(current_path, os.pardir))
print("Parent Directory:", parent_directory)

sys.path.append(parent_directory)
sys.path.append("/home/anibal/microlensing/simulation_Rubin/roman_rubin/")

from ulens_params import microlensing_params

from scipy.interpolate import interp1d
from matplotlib.ticker import AutoMinorLocator

def set_paper_style():
    plt.rcParams.update({
        "figure.figsize": (7.0, 4.5),
        "figure.dpi": 130,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 12,
        "legend.fontsize": 9,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.linewidth": 1.0,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.top": True,
        "ytick.right": True,
        "xtick.minor.visible": True,
        "ytick.minor.visible": True,
        "legend.frameon": True,
        "legend.framealpha": 0.9,
    })

set_paper_style()
# Magnitudes tabuladas
m = np.array([9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23])

# Signal-to-noise para ODI
S_N_ODI = np.array([
    5228.1, 3298.5, 2080.8, 1312.3, 827.1,
    520.4, 326.1, 202.2, 122.5, 70.7,
    37.5, 18.0, 7.9, 3.3, 1.3
])

SN_interp_log_ODI = interp1d(
    m,
    np.log10(S_N_ODI),
    kind="linear",
    bounds_error=True
)

def delta_m(signal_to_noise):
    return (2.5 / np.log(10)) / signal_to_noise

def SN_ODI(mag_value):
    return 10**SN_interp_log_ODI(mag_value)

def sigma_m_ODI(mag_value):
    return delta_m(SN_ODI(mag_value))

# ============================================================
# Ephemerides y parámetros físicos fijos
# ============================================================

path_ephemerides = "/home/anibal/microlensing/simulation_Rubin/roman_rubin/ephemerides/Roman_positions.npy"
ephemerides = np.load(path_ephemerides)

t0 = np.mean(ephemerides[:, 0])

M_planet = float(1 * u.M_earth.to("M_jup"))
M_star = 1

DS = 8000
DL = 4000

params = microlensing_params("a", 0, 0, DL, M_star, M_planet, DS, 5, 0, 0)

thetaE = params.theta_E()
piE = params.piE()

angle = np.pi / 2
piEE = np.cos(angle) * piE
piEN = np.sin(angle) * piE

theta_star = (1 * u.R_sun / (DS * u.pc)) * u.rad.to("mas") * u.mas
rho = (theta_star / thetaE).decompose()
rho_value = float(rho.value)

print("rho =", rho_value)


# ============================================================
# Valores iniciales
# ============================================================

u0_default = rho_value / 4
tE_default = 25.0
q_default = float((1 * u.M_earth / (1 * u.M_sun)).decompose().value)
s_default = 2.0
alpha_default_deg = 45.0

fs1 = 200
fb1 = 0
fs2 = 200
fb2 = 0


# ============================================================
# Función auxiliar: crea evento + modelo para un tE dado
# ============================================================

def build_event_and_model(tE):
    """
    Construye el evento simulado y el modelo USBL para un valor dado de tE.

    El tiempo de observación se redefine cada vez que cambia tE, porque
    las curvas se generan en el intervalo [t0 - 0.1 tE, t0 + 0.1 tE].
    """

    simulated_event = event.Event()
    simulated_event.name = "Simulated"
    simulated_event.ra = 170
    simulated_event.dec = -70

    # Roman: cadencia de 12 minutos
    deltat = 0.01
    t1 = np.arange(
        t0 - deltat * tE,
        t0 + deltat * tE,
        12 / (24 * 60)
    )

    # ODI: cadencia de 20 segundos
    t2 = np.arange(
        t0 - deltat * tE,
        t0 + deltat * tE,
        2 / (24 * 60 * 60)
    )

    lightcurve_simroman = np.c_[
        t1,
        np.full_like(t1, 19.0),
        np.full_like(t1, 0.000000001)
    ]

    lightcurve_simodi = np.c_[
        t2,
        np.full_like(t2, 19.0),
        np.full_like(t2, 0.000000001)
    ]

    tel_earth = telescopes.Telescope(
        name="ODI",
        camera_filter="G",
        lightcurve=lightcurve_simodi.astype(float),
        lightcurve_names=["time", "mag", "err_mag"],
        lightcurve_units=["JD", "mag", "mag"],
        location="Earth"
    )

    simulated_event.telescopes.append(tel_earth)

    tel_space = telescopes.Telescope(
        name="Roman",
        camera_filter="G",
        lightcurve=lightcurve_simroman.astype(float),
        lightcurve_names=["time", "mag", "err_mag"],
        lightcurve_units=["JD", "mag", "mag"],
        location="Space"
    )

    tel_space.spacecraft_name = "Roman"
    tel_space.spacecraft_positions = {
        "astrometry": [],
        "photometry": ephemerides
    }

    simulated_event.telescopes.append(tel_space)

    model = USBL_model.USBLmodel(
        simulated_event,
        origin=["third_caustic", [0, 0]],
        parallax=["None", 0.0]
    )

    model.define_model_parameters()

    return simulated_event, model, t1, t2
# ============================================================
# Construir evento y modelo UNA SOLA VEZ
# ============================================================

tE_fixed = 25.0

simulated_event, model, t1, t2 = build_event_and_model(tE_fixed)

print("tE fijo =", tE_fixed)
print("N Roman =", len(t1))
print("N ODI   =", len(t2))
def A_to_mag(Zp,Fs,A):
    return Zp-2.5*np.log10(Fs*A)
# ============================================================
# Función principal interactiva
# ============================================================
# ============================================================
# Función principal interactiva
# ============================================================
def plot_usbl_interactive(u0, rho, q, s, alpha_deg, save_path=None):
    """
    Grafica la curva USBL para ODI y Roman con estilo de calidad de paper.

    Parámetros variables:
    u0, rho, q, s, alpha_deg.

    tE queda fijo en tE_fixed.
    """

    alpha = np.deg2rad(alpha_deg)

    params_list = [
        float(t0),
        float(u0),
        float(tE_fixed),
        float(rho),
        float(s),
        float(q),
        float(alpha),
        float(fs1),
        float(fb1),
        float(fs2),
        float(fb2)
    ]

    pyLIMA_parameters = model.compute_pyLIMA_parameters(params_list)

    A_earth = model.model_magnification(
        model.event.telescopes[0],
        pyLIMA_parameters
    )

    A_space = model.model_magnification(
        model.event.telescopes[1],
        pyLIMA_parameters
    )

    fs = 6900

    # ========================================================
    # Roman: flujo con ruido usando pyLIMA
    # ========================================================

    observed_flux, err_observed_flux = brightness_transformation.noisy_observations(
        fs * A_space
    )

    model.event.telescopes[1].lightcurve["flux"] = observed_flux
    model.event.telescopes[1].lightcurve["err_flux"] = err_observed_flux
    model.event.telescopes[1].lightcurve["inv_err_flux"] = 1 / err_observed_flux

    lightcurve_magnitude = model.event.telescopes[1].lightcurve_in_magnitude(
        model.event.telescopes[1].lightcurve
    )

    mag_roman = lightcurve_magnitude[:, 1] - 27.4 + 27.6
    mag_err_roman = lightcurve_magnitude[:, 2]

    # ========================================================
    # ODI: magnitud + error tabulado + ruido gaussiano
    # ========================================================

    mag_odi = A_to_mag(27.6, fs, A_earth)
    mag_err_odi = sigma_m_ODI(mag_odi)

    rng = np.random.default_rng(seed=2026)

    mag_odi_noisy = rng.normal(
        loc=mag_odi,
        scale=mag_err_odi
    )

    # ========================================================
    # Tiempo relativo en horas
    # ========================================================

    t_ref = pyLIMA_parameters["t_center"]

    t2_rel_hr = 24.0 * (t2 - t_ref)
    t1_rel_hr = 24.0 * (t1 - t_ref)

    # Para no saturar la figura con miles de barras de error ODI.
    # El modelo se grafica con todos los puntos, pero los datos ODI
    # se muestran de forma reducida.
    max_odi_points_to_show = 10000
    odi_step = max(1, len(t2) // max_odi_points_to_show)

    t2_plot = t2_rel_hr[::odi_step]
    mag_odi_noisy_plot = mag_odi_noisy[::odi_step]
    mag_err_odi_plot = mag_err_odi[::odi_step]

    # ========================================================
    # Figura
    # ========================================================

    fig, ax = plt.subplots(figsize=(7.0, 4.5), constrained_layout=True)

    # Modelo continuo
    ax.plot(
        t2_rel_hr,
        mag_odi,
        color="black",
        linestyle="-",
        linewidth=1.4,
        label="USBL model",
        zorder=4
    )

    # ODI
    ax.errorbar(
        t2_plot,
        mag_odi_noisy_plot,
        yerr=mag_err_odi_plot,
        fmt="o",
        markersize=2.2,
        markerfacecolor="tab:blue",
        markeredgecolor="none",
        ecolor="tab:blue",
        elinewidth=0.5,
        capsize=0,
        alpha=0.35,
        label="ODI / ground",
        rasterized=True,
        zorder=2
    )

    # Roman
    ax.errorbar(
        t1_rel_hr,
        mag_roman,
        yerr=mag_err_roman,
        fmt="v",
        markersize=5.0,
        markerfacecolor="tab:orange",
        markeredgecolor="black",
        markeredgewidth=0.5,
        ecolor="tab:orange",
        elinewidth=0.8,
        capsize=1.5,
        alpha=0.95,
        label="Roman / L2",
        zorder=5
    )

    # Línea vertical en el centro
    ax.axvline(
        0.0,
        color="0.3",
        linestyle="--",
        linewidth=0.9,
        alpha=0.8,
        zorder=1
    )

    ax.set_xlabel(r"Time from $t_{\rm center}$ [hours]")
    ax.set_ylabel("Magnitude")

    # En magnitudes, valores menores son más brillantes
    ax.invert_yaxis()

    # Ticks más prolijos
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())

    ax.tick_params(
        which="major",
        length=5,
        width=1.0,
        direction="in",
        top=True,
        right=True
    )

    ax.tick_params(
        which="minor",
        length=3,
        width=0.8,
        direction="in",
        top=True,
        right=True
    )

    # Grilla sutil
    ax.grid(
        which="major",
        linestyle="-",
        linewidth=0.4,
        alpha=0.25
    )

    # Caja con parámetros, mejor que un título largo
    param_text = (
        rf"$t_E = {tE_fixed:.1f}\,\mathrm{{d}}$" "\n"
        rf"$u_0 = {u0:.1e}$" "\n"
        rf"$\rho = {rho:.1e}$" "\n"
        rf"$q = {q:.1e}$" "\n"
        rf"$s = {s:.2f}$" "\n"
        rf"$\alpha = {alpha_deg:.0f}^\circ$"
    )

    ax.text(
        0.03,
        0.97,
        param_text,
        transform=ax.transAxes,
        fontsize=9,
        va="top",
        ha="left",
        bbox=dict(
            boxstyle="round,pad=0.3",
            facecolor="white",
            edgecolor="0.7",
            alpha=0.9
        )
    )

    ax.legend(
        loc="upper right",
        frameon=True,
        framealpha=0.9,
        borderpad=0.4,
        handlelength=1.8
    )

    # Guardado opcional
    if save_path is not None:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()
    plt.close(fig)

# ============================================================
# Widgets
# ============================================================
# ============================================================
# Widgets
# ============================================================
# ============================================================
# Widgets
# ============================================================

u0_slider = widgets.FloatLogSlider(
    value=max(float(u0_default), 1e-5),
    base=10,
    min=-5,
    max=0,
    step=0.05,
    description=r"u0",
    continuous_update=False,
    readout_format=".2e"
)

rho_slider = widgets.FloatLogSlider(
    value=max(float(rho_value), 1e-8),
    base=10,
    min=-8,
    max=-1,
    step=0.05,
    description=r"rho",
    continuous_update=False,
    readout_format=".2e"
)

q_slider = widgets.FloatLogSlider(
    value=q_default,
    base=10,
    min=-15,
    max=0,
    step=0.05,
    description=r"q",
    continuous_update=False,
    readout_format=".2e"
)

s_slider = widgets.FloatSlider(
    value=s_default,
    min=0.1,
    max=5.0,
    step=0.01,
    description=r"s",
    continuous_update=False,
    readout_format=".2f"
)

alpha_slider = widgets.FloatSlider(
    value=alpha_default_deg,
    min=0.0,
    max=360.0,
    step=1.0,
    description=r"alpha [deg]",
    continuous_update=False,
    readout_format=".1f"
)

tE_label = widgets.HTML(
    value=rf"<b>tE fijo:</b> {tE_fixed:.1f} días"
)

ui = widgets.VBox([
    tE_label,
    u0_slider,
    rho_slider,
    q_slider,
    s_slider,
    alpha_slider
])

out = widgets.interactive_output(
    plot_usbl_interactive,
    {
        "u0": u0_slider,
        "rho": rho_slider,
        "q": q_slider,
        "s": s_slider,
        "alpha_deg": alpha_slider
    }
)

%matplotlib inline
display(ui, out)

Parent Directory: /home/anibal/microlensing/simulation_Rubin/roman_rubin/notebooks
rho = 0.0005761505427078632
tE fijo = 25.0
N Roman = 60
N ODI   = 21600


Output()

In [20]:
(1.9*u.M_earth.to("M_sun"))/(10*u.M_sun)


<Quantity 5.70662976e-07 1 / solMass>

In [8]:
(10e+6)*u.M_jup.to("M_sun")

9545.942339693249

In [35]:
fs=6900
print(27.6-2.5*np.log10(fs))

18.002877273156862
